# Baidu Unlimited-OCR — Free GPU server for local development

This notebook runs the `baidu/Unlimited-OCR` model on Colab's free T4 GPU using SGLang and exposes it via a Cloudflare quick tunnel, so your Next.js app on your laptop can use it as the OCR engine.

**How it works:**
1. Sets up Python 3.12 + a venv.
2. Installs the SGLang wheel from the Baidu repo.
3. Downloads the model from Hugging Face.
4. Starts the SGLang OpenAI-compatible server on port 10000.
5. Exposes it over HTTPS with a free Cloudflare tunnel.
6. Prints a serialized logit-processor string you copy into `.env.local`.

**Keep this Colab tab open while you develop** — when it disconnects, the server stops and the app falls back to Vision/pdf-text.

---

**Run the cells top to bottom (Runtime menu → Run all).**

In [ ]:
# 1) Verify we have a GPU (should show a T4 with ~16GB)
!nvidia-smi

In [ ]:
# 2) Install uv and create a Python 3.12 virtualenv (SGLang requirement)
!pip install -q -U uv
!uv venv /content/unlimited-ocr-venv --python 3.12
!uv pip install --python /content/unlimited-ocr-venv/bin/python -U pip

In [ ]:
# 3) Download the SGLang wheel committed in the Baidu repo
!curl -sL -o /content/sglang.whl https://github.com/baidu/Unlimited-OCR/raw/main/wheel/sglang-0.0.0.dev11416+g92e8bb79e-py3-none-any.whl
!ls -lh /content/sglang.whl

In [ ]:
# 4) Install SGLang + pinned kernels + PyMuPDF (model needs) into the venv.
#    Expected: no CUDA build errors and "successfully installed" at the end.
!uv pip install --python /content/unlimited-ocr-venv/bin/python /content/sglang.whl
!uv pip install --python /content/unlimited-ocr-venv/bin/python kernels==0.11.7
!uv pip install --python /content/unlimited-ocr-venv/bin/python pymupdf==1.27.2.2 huggingface_hub

In [ ]:
# 5) Download the model weights (~6GB) into /content/Unlimited-OCR
/content/unlimited-ocr-venv/bin/python - <<'PY'
from huggingface_hub import snapshot_download
p = snapshot_download('baidu/Unlimited-OCR', local_dir='/content/Unlimited-OCR')
print('Downloaded to:', p)
PY

In [ ]:
# 6) Start the SGLang server in the background on port 10000.
#    NOTE: --attention-backend fa3 is Hopper-only; on the free T4 we let
#    SGLang pick the backend automatically (triton / flashinfer).
!/content/unlimited-ocr-venv/bin/python -c \
    "import torch; print('GPU:', torch.cuda.get_device_name(0)); print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1))"
!nohup /content/unlimited-ocr-venv/bin/python -m sglang.launch_server \
    --model /content/Unlimited-OCR \
    --served-model-name Unlimited-OCR \
    --page-size 1 \
    --mem-fraction-static 0.8 \
    --context-length 32768 \
    --enable-custom-logit-processor \
    --disable-overlap-schedule \
    --skip-server-warmup \
    --host 0.0.0.0 \
    --port 10000 \
    > /content/sglang_server.log 2>&1 &
!echo "server starting, PID $!"

In [ ]:
# 7) Wait until the server is ready (health endpoint returns OK). If it never
#    becomes ready, print the ERROR lines from the server log so we can see why.
import time, urllib.request

ready = False
for i in range(180):
    try:
        with urllib.request.urlopen('http://127.0.0.1:10000/health', timeout=3) as r:
            if r.read() == b'OK':
                print('Server is READY after', i + 1, 'checks')
                ready = True
                break
    except Exception:
        pass
    time.sleep(5)

if not ready:
    print('Server did not become ready. ERROR lines from sglang_server.log:')
    import re
    log = open('/content/sglang_server.log').read()
    errors = [l for l in log.splitlines() if re.search(r'error|traceback|assert|exception', l, re.I)]
    print('\n'.join(errors[-20:]) if errors else log[-1500:])

In [ ]:
# 8) Generate the serialized DeepseekOCRNoRepeatNGramLogitProcessor string.
#    Copy the FULL output (it is a long JSON with a hex payload) into .env.local.
/content/unlimited-ocr-venv/bin/python -c "from sglang.srt.sampling.custom_logit_processor import DeepseekOCRNoRepeatNGramLogitProcessor; print(DeepseekOCRNoRepeatNGramLogitProcessor.to_str())"

In [ ]:
# 9) Install Cloudflare's quick tunnel (no account needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

In [ ]:
# 10) Start the tunnel to expose the local server over HTTPS
!nohup /content/cloudflared tunnel --url http://127.0.0.1:10000 > /content/tunnel.log 2>&1 &
!echo "tunnel starting..."

In [ ]:
# 11) Grab the public HTTPS URL (a https://*.trycloudflare.com address)
import re, time

url = None
for i in range(30):
    try:
        text = open('/content/tunnel.log').read()
        m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', text)
        if m:
            url = m.group(0)
            break
    except FileNotFoundError:
        pass
    time.sleep(2)

print('\n===== YOUR PUBLIC OCR SERVER URL =====')
print(url or 'URL not found yet; check /content/tunnel.log')
print('========================================\n')
print('Paste this into .env.local as:')
print('UNLIMITED_OCR_URL=' + (url or 'https://PASTE_TUNNEL_HERE'))
print('UNLIMITED_OCR_BACKEND=sglang')

In [ ]:
# Optional: quick end-to-end test against the tunnel
# (replace TUNNEL_URL with the value printed above)
TUNNEL_URL = 'PASTE_TUNNEL_URL_HERE'

if TUNNEL_URL.startswith('http'):
    import base64, json, urllib.request
    # tiny 1x1 png
    b64 = 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAIAAACQd1PeAAAADUlEQVQI12P4z8AAAAMBAQDJ/pLvAAAAAElFTkSuQmCC'
    payload = {
        'model': 'Unlimited-OCR',
        'messages': [{
            'role': 'user',
            'content': [
                {'type': 'text', 'text': '<image>document parsing.'},
                {'type': 'image_url', 'image_url': {'url': 'data:image/png;base64,' + b64}},
            ],
        }],
        'temperature': 0,
        'max_tokens': 512,
        'skip_special_tokens': False,
        'images_config': {'image_mode': 'gundam'},
        'stream': True,
    }
    req = urllib.request.Request(
        TUNNEL_URL.rstrip('/') + '/v1/chat/completions',
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json'},
    )
    with urllib.request.urlopen(req, timeout=120) as r:
        out = r.read().decode()
    print('Tunnel OK. Raw stream sample:')
    print(out[:400])
else:
    print('Set TUNNEL_URL to test.')

## Finishing up

### `.env.local` on your laptop
```
UNLIMITED_OCR_URL=https://xxxx-xxxx.trycloudflare.com
UNLIMITED_OCR_BACKEND=sglang
UNLIMITED_OCR_MODEL=Unlimited-OCR
UNLIMITED_OCR_SGLANG_LOGIT_PROCESSOR=<paste the FULL string from step 8>
```

### Try it
```bash
npm run dev
```
Upload an affidavit PDF in the Candidate Profile page, or run the bulk pipeline:
```bash
npm run db:ocr -- --limit=5 --engine=unlimited
```

### Notes
- The Colab tab **must stay open**; the tunnel URL changes every session.
- Free-tier Colab sessions last ~1–2h; re-run all cells to get a fresh URL.
- If the tunnel is unreachable, the app automatically falls back to Google Vision / pdf-text.
- For heavier runs, upgrade to Colab Pro (better GPU) or use the same cells on a GPU machine.

### Keeping the session alive (optional)
Paste into the browser console and leave the tab open to simulate activity:
```js
setInterval(() => { document.querySelector('colab-connect-button').click() }, 60000)
```